# 02. Program or library

**Tier:** Intermediate. **Why you'd use this:** The same file should define `greet` for importers and print a demo only when you run it. `__name__` is how the file tells those two cases apart.

**After this lesson you can:** read a `__main__` guard; tell import output from run output; know why notebooks always look like `__main__`.
About 25 minutes. **Uses:** [Imports](01-imports-lesson.ipynb)


## Quick review

1. Where does Python cache a loaded module?
2. Does `from math import sqrt` skip loading `math`?

<details><summary>Answers</summary>

`sys.modules`. No. It still loads and caches `math`.
</details>


## The idea

When you run a file as a program, its `__name__` is `"__main__"`. When you import it, `__name__` is the module name. Put demo code under `if __name__ == "__main__":` so imports stay quiet. Notebooks always use `"__main__"`, so this lesson uses the real file `greet.py`.

<details><summary>Explain more</summary>

Importing `greet` prints the module name line at top level. Running `python greet.py` also runs the guarded demo.
</details>


## See it work

Read `greet.py`, import it, then run it as a program.


In [1]:
from pathlib import Path

# 1. The file itself shows the guard.
print(Path("greet.py").read_text())

def greet(name: str) -> str:
    return f"hello {name}"


print("module name is", __name__)

if __name__ == "__main__":
    print(greet("world"))



In [2]:
import greet  # noqa: F401

module name is greet


In [3]:
import subprocess
import sys

print(subprocess.run([sys.executable, "greet.py"], capture_output=True, text=True).stdout)

module name is __main__
hello world



## Check yourself

Answer these before you run the cell.

1. What is `__name__` inside an imported `greet`?
2. What is `__name__` when `greet.py` is the program entry?


In [4]:
print(greet.__name__)
print(greet.greet("Ada"))

greet
hello Ada


<details><summary>Answers</summary>

`"greet"`. `"__main__"`.
</details>


## Why it works this way

The interpreter sets `__name__` when it creates the module object. For the entry script, that value is the string `"__main__"`. For an imported file, it is the module's dotted name. The `if __name__ == "__main__":` guard is ordinary string comparison against that value. Deep dive: [The import system](../../internals/09-imports/01-import-system-lesson.ipynb).

Notebooks complicate the lesson because every cell runs in a module already named `"__main__"`. That is why we shell out with `subprocess` to run `greet.py` as a real program. Importing in-process shows the library path: top-level prints still run once, but the guarded block does not.

`python -m package` sets up `sys.path` so package-relative imports work, and still uses `"__main__"` for the entry module. Prefer that form for packages. Prefer `python file.py` for a single script.

Evidence: importing prints `module name is greet` without `hello world`, and the subprocess run prints both the name line as `__main__` and the greeting.

Top-level code outside the guard still runs on import. That is useful for constants and function definitions, and harmful for network calls or argument parsing. Keep import-time work cheap and side-effect free. Put CLI parsing inside the guard or in a `main()` called from the guard.  The guard is not special syntax. It is a comparison you could write by hand against any string. The convention matters because tools and readers expect it. Running via subprocess is evidence, not a different language rule. When a library file starts printing on import, move that print under the guard or delete it.


In [5]:
import subprocess
import sys

import greet

print("imported name", greet.__name__)
print(subprocess.run([sys.executable, "greet.py"], capture_output=True, text=True).stdout)

imported name greet
module name is __main__
hello world



## When to use it

Use a `__main__` guard for demos, CLIs, and scripts that are also importable. Keep libraries quiet on import. Don't put required library setup only inside the guard. Don't trust notebook `__name__` to teach this distinction. Use subprocess or a terminal when you need to prove program versus import behaviour.  Keep `main()` small and testable even when it lives behind the guard.


## Common mistake

Looking for a module that is not on `sys.path`. The error's last line names the missing module.


In [6]:
module_name = "not_a_real_module"
print("trying", module_name)
import not_a_real_module  # noqa: F401, E402

trying not_a_real_module


ModuleNotFoundError: No module named 'not_a_real_module'

Put the file on `sys.path`, or run from the right working directory, or use `python -m` inside a package.


In [7]:
import greet

print(greet.greet("world"))

hello world


## Try it

Add a second print under the `__main__` guard in `greet.py` that shows `sys.argv[0]`. Which value do you see for import versus run?


## Recap

- Import sets `__name__` to the module name. A run sets `"__main__"`.
- Guard demo code with `if __name__ == "__main__":`.
- Notebooks hide the difference; use a real file to see it.

**Next:** [Packages and environments](03-packages-lesson.ipynb)
